#### Опитимизация датасета

### 1. Низкоуровневое хранение и форматы (1–20)
- Выбор бинарного формата: переход с CSV/JSON на Parquet, Feather, Arrow или ORC.
- Использование колоночного хранения: (Parquet/ORC) для эффективного сжатия и чтения только нужных столбцов.
- Использование строкового хранения: (Feather/Arrow IPC) для максимальной скорости чтения строк целиком.
- Выбор алгоритма сжатия: Snappy (скорость), Zstd (баланс), Gzip/Brotli (максимальное сжатие).
- Уровень сжатия (Compression Level): настройка агрессивности алгоритма (например, Zstd level 3 vs 22).
- Разделение на чанки (Row Group Size): оптимизация размера строковых групп в Parquet под размер блока диска или L1/L2 кэша CPU.
- Разбиение на партиции (Partitioning): деление датасета на файлы по значениям (например, по дате или ID региона).
- Использование дельта-кодирования (Delta encoding): хранение разницы между соседними числами вместо абсолютных значений (идеально для ID и таймстампов).
- Использование словарного кодирования (Dictionary encoding): замена повторяющихся строк целыми числами-индексами.
- Run-Length Encoding (RLE): сжатие последовательностей одинаковых значений.
- Bit-packing (упаковка битов): хранение чисел, занимающих меньше 8 бит (например, флаги 0/1), в одном байте.
- Использование Sparse-форматов: для матриц и векторов с большим количеством нулей (CSR, CSC).
- Отказ от индексов: удаление B-tree и Bitmap индексов, если чтение всегда последовательное.
- Кластеризация данных (Data clustering): сортировка данных внутри файла по ключу запроса для улучшения коэффициента сжатия.
- Использование Memory-mapped files (mmap): для работы с датасетами, превышающими объем ОЗУ, без полной загрузки.
- Выбор контейнера хранения: HDF5 для иерархических научных данных, LMDB/SQLite для ключ-значение.
- Отключение выравнивания (Padding): отказ от выравнивания структур в памяти, если это поддерживает формат.
- Использование Endian-сти (Byte Order): выбор Little-Endian для совместимости с современными CPU без конвертации.
- Блочное сжатие против потокового: использование форматов с независимыми блоками для параллельной декомпрессии.
- Встраивание словарей в заголовок файла: избежание внешних файлов метаданных.

### 2. Числовые типы и разрядность (21–40)
- Downcasting целочисленных: int64 -> int32 -> int16 -> int8 (проверка диапазона min/max).
- Использование беззнаковых типов: uint8 вместо int8, если нет отрицательных значений (экономия знакового бита).
- Downcasting чисел с плавающей точкой: float64 -> float32 (основной источник экономии памяти в ML).
- Использование Half-precision (float16): для нейросетей и данных с допустимой потерей точности.
- Использование Brain-float (bfloat16): если архитектура CPU/GPU поддерживает, сохраняет экспоненту float32.
- Fixed-point arithmetic: замена float32 на int32 с фиксированным делителем (например, хранить цену в копейках).
- Масштабирование (Scaling): умножение данных на коэффициент для перевода в больший целочисленный тип с последующей упаковкой.
- Квантование (Quantization): равномерное или логарифмическое сжатие диапазона значений в uint8.
- Логарифмическое преобразование: хранение log(x) вместо x для данных с тяжелым хвостом (экономит биты на больших значениях).
- Сдвиг диапазона (Offset): вычитание min из данных перед упаковкой в меньший тип.
- Агрегация таймстампов: хранение только Unix-времени в секундах (int32) вместо datetime64[ns] (int64).
- Удаление часового пояса: хранение времени в UTC без строкового обозначения зоны.
- Бинаризация порогов: замена непрерывных значений на 0/1 по порогу (Binarization).
- Триномизация: сведение к трем состояниям (-1, 0, 1).
- Использование разреженных массивов (Sparse arrays): для массивов, где >90% значений — нули.
- Отказ от NaN в числах: замена на sentinel-значения (например, -9999) в целочисленных типах (целые типы не тратят биты на сигнализацию NaN).
- Пакетное хранение (Batching): хранение векторов как единого бинарного blob'а вместо массива объектов.
- Использование целочисленного типа для ID: замена UUID (128 бит строкой) на uint64 или uint128.
- Отказ от объектных оберток: использование нативных массивов (NumPy) вместо списков Python.
- Проверка на дублирование точности: если данные измерены с точностью до грамма, нет смысла хранить float64.

### 3. Строковые и категориальные данные (41–60)
- Перевод в категорию (Categorical dtype): замена object/string на целочисленные коды.
- Оптимизация кодов категорий: использование uint8 (до 255 категорий), uint16 (до 65к).
- Анализ кардинальности (Cardinality): метрика — отношение уникальных значений к общему числу строк.
- Отказ от строковых ID: замена хэш-сумм и артикулов на целочисленные суррогатные ключи.
- Сжатие префиксов (Prefix compression): хранение только различающейся части строки.
- N-gram индексация для поиска: вместо хранения полных текстов.
- Замена текста на эмбеддинги: хранение вектора float32[512] вместо текста на 1000 символов (радикальное сжатие объема, но рост вычислительной сложности).
- Замена эмбеддингов на бинарные: использование int8 или бинарных (Binary quantized) эмбеддингов.
- Токенизация и хранение словаря: вынос словаря за пределы таблицы.
- Использование Byte Pair Encoding (BPE): для эффективного хранения текстовых корпусов.
- Усечение строк (String truncation): обрезка до N символов, если хвост не несет информации.
- Нормализация Unicode: приведение к единой форме (NFKC/NFKD) для устранения дубликатов визуально одинаковых символов.
- Удаление регистра: приведение всех строк к lower() для увеличения вероятности попадания в словарь.
- Удаление лишних пробелов и спецсимволов: очистка на этапе ETL.
- Замена локальных путей на хэши: вместо хранения длинных путей к файлам.
- Использование Intern-pool: хранение только одной копии повторяющейся строки в памяти.
- Delta-кодирование для строк: хранение только добавленного суффикса (полезно для иерархических имен).
- Кодирование Хаффмана: на уровне приложения для высокочастотных строк.
- Отказ от экранирования: выбор формата (например, Parquet), не требующего экранирования кавычек и запятых.
- Метрика Fill Rate для строк: доля непустых строк; если она низкая, использовать Sparse-структуру.

### 4. Структура данных и денормализация (61–75)
- Нормализация (Database Normalization): вынос повторяющихся строк в справочники (уменьшает размер основной таблицы).
- Денормализация для колоночного хранения: объединение таблиц в один плоский Parquet-файл (уменьшает оверхед на метаданные и количество файлов).
- Удаление константных столбцов: метрика — Variance = 0.
- Удаление дублирующих столбцов: метрика — Correlation = 1.0.
- Удаление производных признаков: удаление столбцов, которые можно вычислить за O(1) из других (например, total = price * qty).
- Вертикальное партиционирование: разделение таблицы на «горячие» (часто читаемые, легкие) и «холодные» (тяжелые, редкие) столбцы.
- Удаление первичных ключей из тела таблицы: если используется колоночное хранение, PK нужен только для индексации.
- Группировка мелких признаков: объединение нескольких bool-столбцов в один uint8 (битовая маска).
- Разделение широких строк: разбиение JSON-полей на отдельные столбцы до записи в хранилище.
- Удаление метаданных строк: отказ от хранения имени столбца в каждой ячейке (как в JSON).
- Использование схем (Schema enforcement): Avro/Protobuf/Parquet schema для исключения хранения имен типов.
- Удаление индексов по низкой кардинальности: удаление индексов на столбцы с малым числом уникальных значений (например, пол).
- Агрегация на этапе записи: хранение предрассчитанных сумм/средних за день вместо сырых тиковых данных.
- Округление координат: снижение точности GPS с 1.000000 до 1.000 (с метров до километров).
- Удаление временных меток логирования: если они не нужны для анализа.

### 5. Семантическая очистка и фильтрация (76–90)
- Удаление дубликатов строк: drop_duplicates().
- Удаление выбросов (Outlier removal): обрезка по 3 сигма или IQR (уменьшает разброс, повышая эффективность квантования).
- Фильтрация по порогу значимости: удаление признаков с низкой взаимной информацией (Mutual Information) с таргетом.
- Удаление пустых строк и столбцов: метрика — is_empty_ratio.
- Сэмплирование (Downsampling): если датасет избыточен, использование репрезентативной выборки (10% или 1%).
- Унификация единиц измерения: перевод всех валют в одну, всех весов в кг (позволяет сузить диапазон и разрядность).
- Заполнение пропусков (Imputation): замена NaN на -1, 0 или медиану (типизированные данные сжимаются лучше, чем объекты с NaN).
- Удаление шумовых признаков: фильтрация столбцов, состоящих из случайного шума (высокая энтропия, низкая предсказательная сила).
- Удаление PII (персональных данных): хэширование или удаление ФИО, телефонов, если они не нужны для модели.
- Округление дат: усечение до дня/недели/месяца вместо секунд.
- Удаление стоп-слов в текстах: уменьшение длины строк.
- Стирание (Pruning) редких категорий: объединение категорий с частотой < 0.1% в группу "Other".
- Удаление коррелированных признаков (Feature dropping): оставление одного из группы признаков с корреляцией > 0.95.
- Логирование изменений (Data Diff): хранение только дельты изменений датасета, а не полной копии.
- Проверка на информационную энтропию: удаление столбцов с энтропией близкой к максимальной (белый шум).

### 6. Вычислительные прокси-метрики и системные показатели (91–100)
- Memory footprint (Footprint в ОЗУ): DataFrame.memory_usage(deep=True).sum().
- Disk footprint: физический размер файла на диске (ls -lh).
- Compression ratio: Original_Size / Compressed_Size.
- Read/Write throughput (МБ/с): метрика скорости pd.read_parquet() vs pd.read_csv().
- CPU utilization during decompression: процент загрузки ядер при чтении.
- Cache miss rate: при обходе данных (критично при переходе от строкового к колоночному хранению).
- Garbage Collection (GC) pressure: нагрузка на сборщик мусора при работе с объектными типами.
- Time-to-first-byte (TTFB): задержка до начала чтения первого байта из холодного хранилища.
- Overhead of metadata: отношение размера метаданных (схемы, статистики чанков) к размеру полезных данных.
- End-to-end training/inference latency: финальная метрика — насколько быстрее стала работать целевая модель на сжатом датасете.

### 7. Продвинутые алгоритмические и статистические преобразования

- Principal Component Analysis (PCA): проекция данных на главные компоненты (например, сохранение 95% дисперсии) для радикального снижения размерности.
- Incremental PCA: для данных, не помещающихся в память.
- Random Projection (Метод Джонса-Линдена): быстрое случайное уменьшение размерности с гарантией сохранения расстояний.
- Autoencoder Compression: обучение нейросети для сжатия данных в латентное пространство (например, dim 1000 -> dim 32).
- UMAP (Uniform Manifold Approximation and Projection): нелинейное снижение размерности для сохранения локальных структур.
- t-SNE (только для хранения эмбеддингов): крайне агрессивное сжатие в 2D/3D, если важна только визуализация.
- Feature Agglomeration: иерархическая кластеризация признаков и замена кластеров их средним.
- Бинарное квантование (Binary Quantization): преобразование float32 векторов в бинарные (0/1) — сжатие в 32 раза для эмбеддингов.
- Product Quantization (PQ): разбиение векторов на подпространства и квантование каждого отдельно (стандарт в векторных базах данных).
- Optimized Product Quantization (OPQ): предварительный поворот пространства перед PQ для минимизации ошибки.
- Locality-Sensitive Hashing (LSH): замена векторов их хэш-сигнатурами для поиска соседей.
- Non-negative Matrix Factorization (NMF): разложение матрицы на неотрицательные компоненты.
- Dictionary Learning: обучение разреженного словаря атомов, которыми кодируется датасет.
- Wavelet Transform (Дискретное вейвлет-преобразование): сжатие сигналов и временных рядов с отбрасыванием мелких коэффициентов.
- Discrete Cosine Transform (DCT): основа JPEG; применимо к табличным данным с сильной пространственной корреляцией.
- Полиномиальные признаки низкой степени: замена дорогих трансцендентных функций (sin, log) их аппроксимацией.
- Bucketization (Биннинг): замена непрерывного признака на ID интервала (например, возраст 25 -> bin_3).
- Target Encoding: замена категории средним значением таргета (превращает категорию в одно число float32).
- WOE (Weight of Evidence) Encoding: логарифмическое отношение шансов для категорий.
- James-Stein Encoding: сжатое (shrunk) целевое кодирование для борьбы с переобучением на редких категориях.
- Feature Hashing (Hashing Trick): проекция произвольного числа признаков в фиксированный вектор меньшей размерности через хэш-функцию.
- Count Sketching: вероятностная оценка частот элементов в потоке данных.
- Bloom Filters для фильтрации: замена списков ID на компактную вероятностную структуру.
- HyperLogLog: оценка количества уникальных элементов (cardinality) в 1000 раз компактнее точного списка.
- T-Digest: компактная структура для оценки квантилей и перцентилей на потоковых данных.
- KLL Sketch (Kolmogorov-Lazaréff-Lloyd): еще более точная и компактная структура для квантилей.
- Piecewise Aggregate Approximation (PAA): сжатие временных рядов путем усреднения сегментов.
- Symbolic Aggregate approXimation (SAX): преобразование PAA в дискретные символы (строку).
- Перестановка значений (Value Permutation): сортировка столбца для улучшения локальности и сжатия (Parquet делает это автоматически, но можно усилить).
- Delta-of-Delta для таймстампов: хранение изменения изменения времени (часто равно 0 или 1).
- Gorilla Encoding: специализированный алгоритм сжатия для временных рядов (используется в Facebook/Meta TSDB).
- Chimp / Chimp128: современные преемники Gorilla с еще лучшим коэффициентом сжатия.
- Пороговая отсечка весов (Weight Pruning): если датасет — это веса нейросети, обнуление весов ниже порога.
- Structured Pruning: удаление целых нейронов или каналов, а не отдельных весов.
- Quantization-Aware Training (QAT): дообучение модели с учетом будущей низкой разрядности данных.
- Distillation (Дистилляция знаний): перенос знаний из тяжелой модели в легкую, которая требует меньше входных данных.
- Top-K Sparsity: хранение только K самых значимых значений в каждой строке/векторе.
- Eigenvalue Decomposition: сжатие ковариационных матриц.
- Замена float на логарифмическую шкалу (Log-numbers): хранение sign(x) * log(1 + |x|).
- Использование формата fp8 (E4M3/E5M2): если вычислительная среда (GPU) поддерживает 8-битные числа с плавающей точкой.

### 8. Системные, инфраструктурные и метаданные

- Z-Order (Morton Order) кривая: многомерная сортировка данных для улучшения сжатия коррелированных колонок (lat/lon, цена/площадь).
- Hilbert Curve: более сложная версия Z-Order для сохранения локальности.
- Использование Shadow Paging: при записи на диск для избежания двойной записи метаданных.
- Отказ от транзакционных логов (WAL): если датасет только для чтения (Read-only).
- Мемоизация функций трансформации: хранение хэшей входных данных, чтобы не пересчитывать тяжелые фичи.
- On-the-fly computation: полный отказ от хранения производных столбцов, вычисление их в момент запроса (Zero-byte storage).
- Late Materialization: откладывание декодирования тяжелых колонок до последнего момента в плане выполнения запроса.
- Vectorized Execution: использование SIMD-инструкций CPU для декомпрессии (стандарт Arrow/Parquet).
- Prefetching и Read-ahead: системная настройка ОС для чтения сжатых блоков.
- Direct I/O (O_DIRECT): обход кэша ОС для предотвращения двойного хранения сжатых данных в памяти.
- Отказ от выравнивания структур (Packed structs): в кастомных бинарных форматах.
- Использование Compact Protocol (Thrift/Protobuf): вместо JSON для передачи данных между сервисами.
- Zero-copy Deserialization: использование форматов, которые можно читать напрямую из mmap без аллокации.
- Метрика Physical-to-Logical size ratio: насколько данные раздуты служебной информацией файловой системы.
- Inode density: уменьшение количества мелких файлов (объединение в один большой Parquet).
- Data lineage pruning: удаление истории происхождения данных, если она не нужна для аудита.
- Удаление статистик чанков (Min/Max per chunk): если используется внешнее индексирование (хотя это вредит скорости чтения).
- Использование Column-level CHECK constraints: позволяет СУБД не хранить заведомо невалидные (и часто избыточные) данные.
- Data Vault 2.0 Satellites с эффектом сжатия: хранение только изменений (дельт) атрибутов во времени.
- CDC (Change Data Capture) логи вместо снапшотов: хранение потока изменений вместо полной копии датасета.

### 9. Специфические доменные трансформации (финансы, тексты, сигналы)

- Хранение цен в тиках (Ticks): вместо float хранить минимальное изменение цены (целое число).
- Volume aggregation: замена всех сделок за миллисекунду одной агрегированной.
- OHLC Binning: замена тиковых данных на свечи (Open/High/Low/Close) с фиксированным шагом.
- Импутация Bid-Ask спреда: заполнение пропущенных котировок ближайшим значением (уменьшает энтропию).
- Удаление микро-структурных шумов: фильтрация сделок, не изменивших цену (wash trades).
- Словарь финансовых инструментов: замена тикеров (10 байт) на uint16 ID.
- Сжатие новостных лент: удаление HTML-тегов, CSS и скриптов, хранение только чистого текста.
- Stop-word removal и Stemming/Lemmatization: радикальное уменьшение длины текстовых документов.
- Image thumbnails: замена оригинальных изображений (МБ) на миниатюры (КБ) или векторы признаков.
- Chroma Subsampling (4:2:0): для цветных изображений, если яркость важнее цвета.
- Удаление EXIF/GPS метаданных из изображений: очистка оверхеда.
- Аудиокодеки с потерями (Opus/AAC): вместо WAV/PCM.
- Снижение частоты дискретизации (Downsampling audio): с 48 кГц до 16 кГц, если не нужен ультразвук.
- Моно вместо стерео: если пространственный звук не важен.
- VAD (Voice Activity Detection): удаление пауз из аудиозаписей.
- Спектрограммы вместо сырых волн: хранение частотного представления (меньше данных для ML).
- Замена гео-координат на H3 индексы (Uber): вместо lat/lon (2xfloat64) хранить uint64 ячейку.
- Замена гео-координат на S2 (Google): аналогично H3.
- Удаление промежуточных узлов из JSON: сплющивание (flattening) глубоких структур в один уровень.
- Сериализация списков как битовых масок: вместо [1, 5, 10] хранить 0b1000000001000010.

### 10. Метрики оценки качества сжатия и профилирования

- Distortion (Искажение): L2-норма разницы между исходным и восстановленным вектором.
- PSNR (Peak Signal-to-Noise Ratio): классическая метрика для изображений и сигналов.
- Structural Similarity Index (SSIM): перцептивная метрика качества.
- Information Loss (Потеря информации): падение Accuracy/F1-score модели после сжатия данных.
- Mutual Information Retention: какой процент взаимной информации с таргетом сохранился после сжатия.
- Kolmogorov Complexity (оценочно): длина минимальной программы, генерирующей данные (стремление к нему).
- Data Smell (Запах данных): метрика аномалий, указывающих на избыточность (например, 99% одинаковых значений).
- Sparsity Ratio: Count(Non-Zero) / Total_Count.
- N-gram Repetition Rate: частота повторяющихся последовательностей в текстах/логах.
- Birthday Paradox Collisions: вероятность коллизий при Feature Hashing (оценка риска потери информации).
- Condition Number: оценка устойчивости матрицы данных; сжатие часто улучшает число обусловленности.
- Effective Rank: реальное число значимых сингулярных чисел матрицы.
- Data Redundancy Ratio: 1 - (Entropy / Max_Entropy).
- Read Latency P99: 99-й перцентиль задержки чтения после сжатия.
- Cold Start Time: время запуска пайплайна с загрузкой сжатого датасета.
- Memory Fragmentation: оценка дефрагментации ОЗУ при загрузке множества мелких сжатых файлов.
- Storage Cost per Feature: финансовая метрика стоимости хранения одного столбца.
- Energy Efficiency (Дж/ГБ): энергопотребление CPU/GPU на обработку сжатого объема.
- Schema Evolution Cost: сложность изменения схемы максимально сжатого (бинарного) датасета.
- Reversibility Flag: бинарный показатель — можно ли из сжатой формы получить 100% исходных байтов (Lossless) или нет (Lossy).

----

## 1. Что именно должен выдавать агент по промпту

Агент должен не «угадать одну модель», а вернуть структуру:

```json
{
  "task_type": "regression | classification | ranking | clustering | forecasting | anomaly_detection | generation | detection | segmentation | recommendation | RL | causal_inference",
  "target": "что предсказываем/находим/ранжируем",
  "input_modality": ["tabular", "text", "image", "audio", "video", "time_series", "graph", "geospatial", "multimodal"],
  "supervision": "supervised | unsupervised | semi-supervised | self-supervised | RL",
  "model_families": [
    {"name": "gradient_boosting", "score": 0.84},
    {"name": "linear_model", "score": 0.61}
  ],
  "evaluation_metrics": ["MAE", "RMSE", "R2"],
  "constraints": {
    "latency": "high | medium | low",
    "explainability": "required | optional",
    "privacy": "required | optional",
    "scale": "small | medium | high"
  },
  "missing_info": ["не указан объём данных", "не указан тип target"],
  "assumptions": ["предполагаю табличные данные", "target — числовая цена"],
  "confidence": 0.78,
  "evidence": ["предсказать цену", "есть таблица с признаками"]
}
```

Ключевая идея: **промпт → набор признаков → multi-label классификация ML-задачи → retrieval/ranking моделей из каталога → объяснение с evidence**.

---

## 2. Полный список метрик анализа промпта

### A. Структурно-лингвистические метрики промпта

| Метрика | Что показывает |
|---|---|
| `char_count`, `token_count`, `sentence_count` | Размер и детализацию промпта |
| `language`, `code_switching_score` | Язык и смешение языков |
| `readability_score` | Сложность формулировки |
| `imperative_ratio` | Доля повелительных конструкций: «сделай», «предскажи» |
| `question_ratio` | Есть ли в промпте вопросы вместо требований |
| `example_count` | Наличие примеров вход/выход |
| `number_count` | Числа, объёмы, SLA, проценты |
| `negation_count` | Отрицания и ограничения: «не должно», «без» |
| `modality_strength` | «нужно», «можно», «обязательно», «желательно» |
| `specificity_score` | Насколько промпт конкретен |
| `ambiguity_score` | Неоднозначность формулировок |
| `contradiction_score` | Противоречия внутри промпта |
| `jargon_density` | Плотность технических терминов |
| `named_entity_count` | Сущности: товар, клиент, заказ, изображение |
| `technology_mention_count` | Упоминания технологий: Python, SQL, GPU, API |
| `format_marker_count` | JSON, CSV, API, webhook, таблица |
| `domain_keyword_hits` | Попадания в домены: e-commerce, medtech, edtech, finance |

### B. Метрики постановки ML-задачи

| Метрика | Что показывает |
|---|---|
| `task_type_score[classification]` | Есть ли признаки классификации |
| `task_type_score[regression]` | Есть ли «предсказать число/цену/спрос» |
| `task_type_score[ranking]` | «ранжировать», «рекомендовать», «топ-10» |
| `task_type_score[clustering]` | «сгруппировать», «сегментировать», «похожие» |
| `task_type_score[forecasting]` | «прогноз на будущее», «временной ряд» |
| `task_type_score[anomaly_detection]` | «аномалии», «мошенничество», «выбросы» |
| `task_type_score[generation]` | «сгенерировать текст/код/изображение» |
| `task_type_score[detection]` | «найти объект на изображении» |
| `task_type_score[segmentation]` | «сегментировать изображение» |
| `task_type_score[recommendation]` | «рекомендовать товары/контент» |
| `task_type_score[RL]` | «управлять», «оптимизировать стратегию» |
| `task_type_score[causal_inference]` | «оценить эффект воздействия» |
| `supervision_score` | Есть ли разметка: supervised / unsupervised / self-supervised |
| `label_availability_score` | Указано ли, откуда брать labels |
| `target_presence` | Явно ли указан target |
| `target_type` | binary / multiclass / continuous / sequence / graph / ranking |
| `output_type` | Класс, число, вероятность, топ-k, текст, маска |
| `multi_label_flag` | Один объект → несколько меток |
| `imbalance_hint_score` | «редкий класс», «мошенничество», «аномалии» |

### C. Метрики данных и модальности

| Метрика | Что показывает |
|---|---|
| `modality_score[tabular]` | Табличные данные |
| `modality_score[text]` | Текст |
| `modality_score[image]` | Изображения |
| `modality_score[audio]` | Аудио |
| `modality_score[video]` | Видео |
| `modality_score[time_series]` | Временные ряды |
| `modality_score[graph]` | Графы, связи |
| `modality_score[geospatial]` | Геоданные |
| `modality_score[multimodal]` | Несколько модальностей |
| `data_volume_hint` | «тысячи», «миллионы», «терабайты» |
| `feature_count_hint` | Сколько признаков ожидается |
| `temporal_hint` | Есть ли время/дата/период |
| `spatial_hint` | Есть ли геолокация |
| `streaming_hint` | Реалтайм/поток |
| `privacy_sensitivity_score` | Персональные данные, медицина, финансы |
| `data_quality_risk` | Пропуски, шум, разметка «на глаз» |

### D. Метрики ограничений и требований

| Метрика | Что показывает |
|---|---|
| `latency_requirement` | «мгновенно», «реалтайм», «<100 мс» |
| `throughput_requirement` | RPS, одновременные пользователи |
| `cost_sensitivity` | «дёшево», «стартап», «бюджет» |
| `explainability_requirement` | «объяснить решение», «почему» |
| `privacy_requirement` | «конфиденциально», «152-ФЗ», «персональные данные» |
| `on_prem_requirement` | «локально», «без облака», «на своём сервере» |
| `interpretability_requirement` | Нужны ли простые модели |
| `robustness_requirement` | «не должно ломаться», «устойчиво» |
| `fairness_requirement` | «без дискриминации», «равные условия» |
| `deployment_requirement` | batch / online / edge / mobile |

### E. Метрики неопределённости и риска

| Метрика | Что показывает |
|---|---|
| `missing_critical_fields` | Сколько ключевых полей не указано |
| `assumption_count` | Сколько допущений пришлось сделать |
| `ambiguity_score` | Неоднозначность промпта |
| `contradiction_score` | Противоречия |
| `underspecification_score` | Промпт слишком общий |
| `overspecification_score` | Слишком много лишних деталей |
| `risk_score` | Риск ошибочной рекомендации |
| `confidence` | Итоговая уверенность агента |
| `abstention_flag` | Нужно ли воздержаться от точного ответа |

### F. Метрики выбора модели и объяснения

| Метрика | Что показывает |
|---|---|
| `model_family_score[linear]` | Линейные модели |
| `model_family_score[tree]` | Деревья решений |
| `model_family_score[boosting]` | CatBoost/XGBoost/LightGBM |
| `model_family_score[svm]` | SVM |
| `model_family_score[knn]` | kNN/эмбеддинги |
| `model_family_score[neural_net]` | Нейросети общего назначения |
| `model_family_score[cnn]` | CNN для изображений |
| `model_family_score[rnn_transformer]` | RNN/Transformer для текста/рядов |
| `model_family_score[gnn]` | Графовые сети |
| `model_family_score[recommender]` | ALS, matrix factorization, two-tower |
| `model_family_score[anomaly]` | Isolation Forest, AutoEncoder |
| `model_family_score[llm_finetune]` | Дообучение LLM |
| `model_family_score[rag]` | RAG вместо обучения |
| `top_k_margin` | Насколько top-1 модель уверенно отрывается |
| `evidence_coverage` | Доля полей, подкреплённых цитатами из промпта |
| `recommended_metric_set` | Набор метрик качества для будущей модели |
| `baseline_complexity` | Минимальная сложность модели |
| `data_requirement_estimate` | Сколько данных нужно |

---

## 3. Метрики качества самого агента-анализатора

Это уже оценка вашего «Prompt → ML Model Router».

| Метрика | Что измеряет |
|---|---|
| `field_extraction_precision` | Точность извлечения полей |
| `field_extraction_recall` | Полнота извлечения полей |
| `field_extraction_f1` | F1 по полям |
| `task_classification_accuracy` | Точность определения типа ML-задачи |
| `task_classification_macro_f1` | Macro-F1 по типам задач |
| `model_recommendation_top1_accuracy` | Попала ли правильная модель в top-1 |
| `model_recommendation_top3_accuracy` | Попала ли в top-3 |
| `model_recommendation_top5_accuracy` | Попала ли в top-5 |
| `MRR` | Mean Reciprocal Rank для рекомендованных моделей |
| `NDCG@k` | Качество ранжирования моделей |
| `confidence_calibration_ECE` | Насколько confidence соответствует реальной точности |
| `Brier_score` | Калибровка вероятностей |
| `hallucination_rate` | Доля выдуманных моделей/метрик/ограничений |
| `evidence_attribution_accuracy` | Правильно ли указаны цитаты-обоснования |
| `consistency_under_paraphrase` | Стабильность при перефразировании промпта |
| `stability_across_seeds` | Стабильность при повторных запусках |
| `coverage` | Доля промптов, где агент дал уверенный ответ |
| `abstention_rate` | Доля обоснованных отказов |
| `latency_p50/p95/p99` | Скорость ответа агента |
| `cost_per_prompt` | Стоимость анализа одного промпта |

---

## 4. Метрики будущей ML-модели, которые агент должен предложить

Это не метрики промпта, а **метрики, которые агент обязан выбрать под задачу**.

| Тип задачи | Метрики |
|---|---|
| Классификация | Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, LogLoss, Confusion Matrix |
| Регрессия | MAE, MSE, RMSE, MAPE, sMAPE, R², Quantile Loss |
| Ранжирование / рекомендации | NDCG@k, MAP@k, MRR, Recall@k, HitRate@k, Coverage, Diversity |
| Кластеризация | Silhouette, Davies-Bouldin, Calinski-Harabasz, ARI, NMI |
| Прогнозирование рядов | MAE, RMSE, MAPE, MASE, Pinball Loss |
| Детекция / сегментация | IoU, Dice, mAP, Precision/Recall по объектам |
| Генерация текста | BLEU, ROUGE, METEOR, BERTScore, Perplexity, Human Eval |
| Аномалии | Precision@k, Recall, F1, AUROC, Detection Delay |
| RL | Reward, Regret, Sample Efficiency |
| Безопасность / fairness | Bias metrics, Fairness gap, Robustness under attack |

---

## 5. Минимальный MVP-набор метрик

Если делать быстро, начните с 12 метрик:

1. `task_type_score` — multi-label классификация типа ML-задачи.  
2. `target_presence` + `target_type`.  
3. `modality_score` — tabular/text/image/audio/video/time_series.  
4. `supervision_score` — есть ли разметка.  
5. `data_volume_hint` — объём данных.  
6. `latency_requirement`.  
7. `privacy_requirement`.  
8. `model_family_score` — top-3 семейства моделей.  
9. `recommended_metric_set` — метрики качества.  
10. `missing_critical_fields` + `assumption_count`.  
11. `confidence` + `evidence_coverage`.  
12. `model_recommendation_top3_accuracy` — качество агента.

---

## 6. Пример маппинга «фраза в промпте → модель»

| Фраза в промпте | ML-задача | Модель | Метрики |
|---|---|---|---|
| «предсказать цену» | Регрессия | Gradient Boosting / Linear | MAE, RMSE, R² |
| «определить спам» | Бинарная классификация | Logistic Regression / CatBoost / Transformer | F1, PR-AUC |
| «сгруппировать клиентов» | Кластеризация | KMeans / HDBSCAN | Silhouette, ARI |
| «рекомендовать товары» | Ранжирование | ALS / Two-Tower | NDCG@k, MAP@k |
| «прогноз спроса» | Forecasting | Prophet / LightGBM / LSTM | MAPE, RMSE |
| «найти аномалии» | Anomaly Detection | Isolation Forest / AutoEncoder | Precision@k, Recall |
| «сгенерировать ответ» | Генерация | LLM / RAG / Fine-tune | BLEU/ROUGE/BERTScore/Human Eval |
| «найти объект на фото» | Detection | YOLO / Faster R-CNN | mAP, IoU |
| «сегментировать изображение» | Segmentation | U-Net / Mask R-CNN | IoU, Dice |
| «управлять стратегией» | RL | PPO / DQN | Reward, Regret |

## 7. Семантические и прагматические метрики промпта

Это метрики «что клиент *имеет в виду*», а не «какие слова написал».

| Метрика | Что показывает |
|---|---|
| `intent_entropy` | Насколько размыт intent (одна задача или букет) |
| `intent_decomposition_count` | Сколько подзадач в одном промпте |
| `primary_intent_confidence` | Уверенность в главном intent |
| `secondary_intent_ratio` | Доля второстепенных задач |
| `goal_verb_class` | predict / classify / rank / generate / optimize / detect / explain |
| `stakeholder_hint` | Кто пользователь: бизнес, аналитик, разработчик, конечный клиент |
| `business_outcome_hint` | Рост выручки / снижение оттока / экономия / комплаенс |
| `decision_hint` | Решение автоматическое или рекомендательное |
| `actionability_score` | Насколько из промпта ясно, какое действие последует за предсказанием |
| `cost_of_error_hint` | «критично ошибиться», «дёшево ошибиться», «штраф за ошибку» |
| `human_in_the_loop_hint` | «оператор проверяет», «модератор утверждает» |
| `regulatory_scope_hint` | GDPR, 152-ФЗ, HIPAA, PCI DSS, AI Act |
| `sector_hint` | Fintech / retail / medtech / edtech / adtech / industrial |
| `urgency_hint` | «срочно», «через месяц», «MVP к пятнице» |
| `budget_hint` | «дёшево», «enterprise», «без затрат на GPU» |
| `team_maturity_hint` | «новичок», «есть Data Scientist», «нет ML-команды» |
| `build_vs_buy_hint` | «своя модель» vs «готовое API» vs «AutoML» |
| `embedding_centroid_distance` | Близость промпта к эталонным кластерам задач |
| `semantic_entropy` | Энтропия эмбеддинга: размытость смысла |
| `topic_drift_within_prompt` | Смещение темы внутри одного промпта |
| `coreference_quality` | Насколько понятно, к чему относятся местоимения («это», «его») |
| `ellipsis_score` | Пропуски слов, которые надо додумать |

## 8. Метрики извлечения требований (requirement extraction)

| Метрика | Что показывает |
|---|---|
| `functional_requirement_count` | Сколько функциональных требований явно выделено |
| `nonfunctional_requirement_count` | NFR: latency, availability, privacy |
| `constraint_type_distribution` | Технические / бизнес / юридические / UX |
| `hard_vs_soft_constraint_ratio` | «обязательно» vs «желательно» |
| `priority_ambiguity` | Неясно, что важнее при конфликте |
| `conflict_count` | Явные противоречия между требованиями |
| `requirement_completeness_score` | Доля стандартных полей ML-бриф, покрытых промптом |
| `requirement_traceability_score` | Каждое требование → цитата из промпта |
| `implicit_requirement_count` | Требования, выведенные из контекста |
| `assumption_risk_score` | Насколько допущения рискованны |
| `clarification_need_score` | Насколько нужны уточнения (даже если агент не задаёт вопросы) |
| `template_coverage` | Покрытие стандартного чек-листа ML-брифа |

## 9. Data-readiness метрики (агент должен уметь их оценить)

| Метрика | Что показывает |
|---|---|
| `data_source_mentioned` | Указано ли, откуда данные |
| `data_format_mentioned` | CSV / JSON / SQL / images / logs / events |
| `data_freshness_hint` | realtime / daily / historical |
| `data_ownership_hint` | own / partner / public / scraped |
| `label_source_hint` | manual / logs / feedback / weak / none |
| `label_noise_risk` | «разметили на глаз», «разные люди», «нет гайдлайна» |
| `class_balance_hint` | «редкие мошенники», «50/50» |
| `feature_leakage_risk` | Признаки, которых не будет в проде |
| `target_leakage_risk` | Слишком хорошие признаки |
| `temporal_leakage_risk` | Перемешивание времени |
| `sampling_bias_hint` | «только Москва», «только активные» |
| `dataset_size_estimate` | Оценка объёма по числам в промпте |
| `labeling_cost_estimate` | Сколько будет стоить разметка |
| `synthetic_data_need_score` | Нужны ли синтетические данные |
| `pii_presence_score` | Есть ли PII в описании данных |
| `data_drift_risk` | Промпт намекает на быстро меняющийся домен |

## 10. Feature-engineering и baseline-метрики

| Метрика | Что показывает |
|---|---|
| `feature_engineering_complexity` | Насколько сложны признаки |
| `text_feature_need` | TF-IDF / embeddings / BM25 |
| `image_feature_need` | CNN / ViT / CLIP |
| `time_feature_need` | lags / rolling / Fourier |
| `graph_feature_need` | node2vec / GNN |
| `geo_feature_need` | H3 / geohash / PostGIS |
| `categorical_encoding_need` | one-hot / target / CatBoost native |
| `baseline_recommendation` | majority / linear / heuristic |
| `baseline_expected_score` | Ожидаемое качество простого бейзлайна |
| `lift_over_baseline_needed` | Насколько модель должна обогнать бейзлайн |
| `feature_store_need` | Нужен ли feature store |
| `online_feature_need` | Нужны ли признаки в реалтайме |

## 11. Model-selection метрики (глубже, чем top-k)

| Метрика | Что показывает |
|---|---|
| `model_card_match_score` | Совпадение требований с моделью из каталога |
| `data_size_model_fit` | Хватит ли данных для выбранной модели |
| `compute_budget_fit` | Влезает ли модель в бюджет GPU |
| `inference_latency_fit` | Укладывается ли модель в SLA |
| `explainability_fit` | Соответствует ли модель требованию объяснимости |
| `regulatory_approvability` | Можно ли использовать модель в регуляторике |
| `pretrained_availability` | Есть ли готовая предобученная модель |
| `library_maturity_score` | Зрелость библиотеки (sklearn, LightGBM, HF) |
| `deployment_complexity_score` | Сложность деплоя |
| `vendor_lockin_risk` | Риск привязки к вендору |
| `license_risk` | GPL / commercial / open weights |
| `model_size_mb` | Размер артефакта |
| `edge_feasibility_score` | Можно ли на edge |
| `few_shot_capability` | Достаточно ли 10–100 примеров |
| `multilingual_fit` | Нужна ли мультиязычность |

## 12. Метрики риска и безопасности промпта

| Метрика | Что показывает |
|---|---|
| `prompt_injection_score` | Промпт пытается инжектить команды |
| `jailbreak_attempt_score` | Попытка обхода ограничений |
| `data_exfiltration_hint` | «выведи все данные», «покажи БД» |
| `pii_in_prompt` | В промпте есть персональные данные |
| `sensitive_domain_flag` | Медицина, финансы, дети, оружие |
| `harmful_intent_score` | Потенциально вредная задача |
| `compliance_violation_risk` | Нарушение 152-ФЗ / GDPR / AI Act |
| `copyright_risk` | Генерация на чужих данных |
| `toxicity_hint` | Токсичный контент |
| `bias_risk_score` | Признаки чувствительных атрибутов |
| `adversarial_mention` | «обойти детектор», «перехитрить модель» |
| `red_team_priority` | Приоритет красной команды |

## 13. LLM-specific и RAG-метрики (если агент сам на LLM)

| Метрика | Что показывает |
|---|---|
| `context_utilization` | Использован ли весь контекст |
| `context_window_need` | Хватит ли окна |
| `chunking_quality` | Качество чанкинга |
| `retrieval_precision@k` | Точность поиска |
| `retrieval_recall@k` | Полнота поиска |
| `context_relevance` | Релевантность найденного |
| `faithfulness` | Ответ соответствует контексту |
| `answer_relevance` | Ответ отвечает на вопрос |
| `groundedness` | Все утверждения подкреплены источниками |
| `citation_accuracy` | Цитаты правильные |
| `self_consistency` | Согласованность между запусками |
| `chain_of_thought_quality` | Качество рассуждения |
| `tool_call_accuracy` | Правильно ли выбраны tools |
| `tool_call_efficiency` | Нет ли лишних вызовов |
| `planning_score` | Качество плана |
| `step_count` | Длина траектории агента |
| `loops_detected` | Зацикливания |
| `task_success_rate` | Доля успешных траекторий |
| `trajectory_efficiency` | Короткий ли путь к ответу |

## 14. MLOps и lifecycle-метрики (что агент должен предусмотреть)

| Метрика | Что показывает |
|---|---|
| `retraining_frequency_need` | Как часто переобучать |
| `monitoring_need` | Нужен ли drift-monitoring |
| `drift_detection_type` | data / concept / label drift |
| `shadow_mode_need` | Нужен ли shadow-режим |
| `ab_test_need` | Нужен ли A/B |
| `canary_need` | Нужен ли canary |
| `rollback_complexity` | Сложность отката модели |
| `versioning_need` | Нужен ли model registry |
| `reproducibility_need` | Нужна ли воспроизводимость |
| `ci_cd_ml_need` | Нужен ли MLOps-пайплайн |
| `feature_monitoring_need` | Мониторинг фичей |
| `prediction_monitoring_need` | Мониторинг предсказаний |
| `feedback_loop_need` | Нужна ли петля обратной связи |
| `human_review_need` | Нужен ли ручной ревью |
| `data_contract_need` | Нужен ли data contract |

## 15. Business-impact метрики (агент должен их прогнозировать)

| Метрика | Что показывает |
|---|---|
| `expected_business_kpi` | Revenue / churn / conversion / cost |
| `roi_estimate` | Ожидаемый ROI |
| `time_to_value_estimate` | Сколько до первого эффекта |
| `implementation_cost_estimate` | Стоимость внедрения |
| `tco_estimate` | Total Cost of Ownership |
| `risk_adjusted_value` | Value с учётом рисков |
| `expected_lift` | Ожидаемый прирост метрики |
| `cost_of_false_positive` | Цена FP в бизнесе |
| `cost_of_false_negative` | Цена FN в бизнесе |
| `automation_rate` | Доля автоматизации |
| `adoption_risk` | Риск, что не приживётся |
| `change_management_need` | Нужно ли менять процессы |

## 16. Специфические метрики по модальностям

| Модальность | Дополнительные метрики |
|---|---|
| Tabular | `missing_rate`, `cardinality`, `skewness`, `outlier_rate` |
| Text | `token_len`, `vocab_size`, `OOV_rate`, `language_mix`, `domain_specificity` |
| Image | `resolution`, `class_count`, `augmentation_need`, `domain_shift_risk` |
| Audio | `sample_rate`, `snr`, `speaker_diarization_need`, `noise_level` |
| Video | `fps`, `duration`, `scene_complexity`, `temporal_alignment_need` |
| Time series | `seasonality`, `stationarity`, `horizon`, `granularity`, `missing_ts_rate` |
| Graph | `node_count`, `edge_density`, `homophily`, `temporal_graph_need` |
| Geospatial | `resolution`, `coordinate_system`, `tile_count`, `geo_privacy_need` |

## 17. Метрики оценки самого агента (расширенные)

| Метрика | Что показывает |
|---|---|
| `exact_match` | Точное совпадение с эталоном |
| `semantic_similarity` | Косинусная близость к эталону |
| `structured_output_validity` | Валиден ли JSON |
| `schema_adherence` | Соответствие схеме |
| `field_level_f1` | F1 по каждому полю отдельно |
| `task_type_macro_f1` | Macro-F1 по типам ML-задач |
| `task_type_weighted_f1` | Weighted-F1 |
| `multi_label_subset_accuracy` | Точное совпадение всего набора |
| `hamming_loss` | Доля неверных меток |
| `jaccard_score` | Пересечение предсказанных и эталонных меток |
| `ranking_mrr` | Mean Reciprocal Rank моделей |
| `ranking_ndcg@k` | NDCG рекомендованных моделей |
| `ranking_precision@k` | Precision@k |
| `metric_recommendation_f1` | F1 по рекомендованным метрикам |
| `assumption_precision` | Точность сделанных допущений |
| `assumption_recall` | Полнота сделанных допущений |
| `missing_info_recall` | Нашёл ли все критичные пропуски |
| `missing_info_precision` | Не выдумал ли лишних пропусков |
| `confidence_ECE` | Expected Calibration Error |
| `confidence_MCE` | Maximum Calibration Error |
| `reliability_diagram_area` | Площадь под диаграммой надёжности |
| `selective_accuracy` | Точность при abstention-пороге |
| `risk_coverage_curve` | Кривая риск–покрытие |
| `abstention_quality` | Правильно ли агент отказывается |
| `evidence_citation_precision` | Точность цитат |
| `evidence_citation_recall` | Полнота цитат |
| `paraphrase_consistency` | Стабильность при перефразе |
| `multilingual_consistency` | Стабильность между языками |
| `adversarial_robustness` | Устойчивость к adversarial-промптам |
| `counterfactual_stability` | Стабильность при смене деталей |
| `error_severity_distribution` | Насколько критичны ошибки |
| `worst_case_latency` | Худшая задержка |
| `cost_per_correct_answer` | Стоимость правильного ответа |

## 18. Метрики «зрелости» промпта (prompt maturity score)

| Метрика | Что показывает |
|---|---|
| `target_clarity` | Ясность target |
| `label_clarity` | Ясность разметки |
| `data_clarity` | Ясность данных |
| `constraint_clarity` | Ясность ограничений |
| `success_definition_clarity` | Ясность критерия успеха |
| `scope_clarity` | Ясность границ |
| `stakeholder_clarity` | Ясность заказчика |
| `budget_clarity` | Ясность бюджета |
| `timeline_clarity` | Ясность сроков |
| `integration_clarity` | Ясность интеграций |
| `overall_prompt_maturity` | Интегральный скор зрелости |

## 19. Метрики сравнения моделей (model comparison)

| Метрика | Что показывает |
|---|---|
| `wilcoxon_p_value` | Значимость разницы моделей |
| `bootstrap_ci_low/high` | Доверительный интервал |
| `mcnemar_p_value` | Для классификации |
| `effect_size` | Величина разницы |
| `statistical_power` | Хватает ли данных для вывода |
| `cross_validation_variance` | Разброс между фолдами |
| `seed_variance` | Разброс между сидами |
| `ensemble_gain` | Даёт ли ансамбль прирост |
| `blending_gain` | Даёт ли blending прирост |
| `stacking_gain` | Даёт ли stacking прирост |
| `distillation_need` | Нужна ли дистилляция |
| `quantization_need` | Нужна ли квантовая компрессия |

## 20. Метрики для мультиагентных / tool-use сценариев

| Метрика | Что показывает |
|---|---|
| `agent_handoff_count` | Сколько передач между агентами |
| `tool_selection_accuracy` | Правильно ли выбран инструмент |
| `tool_argument_accuracy` | Правильно ли заполнены аргументы |
| `tool_call_failure_rate` | Доля неудачных вызовов |
| `plan_step_count` | Длина плана |
| `plan_redundancy` | Дублирование шагов |
| `plan_completion_rate` | Доля выполненных шагов |
| `replan_count` | Сколько раз перепланировал |
| `task_abandonment_rate` | Доля брошенных задач |
| `human_escalation_rate` | Доля эскалаций человеку |
| `multi_agent_coherence` | Согласованность агентов |
| `shared_memory_utilization` | Использование общей памяти |

---

## 21. Что добавить в MVP-второй волны

К 12 метрикам из первого ответа добавьте:

13. `intent_decomposition_count` — сколько задач в промпте.  
14. `constraint_type_distribution` — какие ограничения и в каком соотношении.  
15. `missing_info_recall/precision` — нашёл ли все пропуски, не выдумал ли лишних.  
16. `assumption_risk_score` — насколько рискованны допущения.  
17. `data_readiness_score` — данные вообще готовы к ML.  
18. `labeling_cost_estimate` — стоимость разметки.  
19. `model_card_match_score` — соответствие модели требованиям.  
20. `confidence_ECE` — откалиброван ли confidence.  
21. `paraphrase_consistency` — стабильность при перефразе.  
22. `structured_output_validity` — валиден ли JSON.  
23. `evidence_citation_precision/recall` — качество цитат.  
24. `worst_case_latency` и `cost_per_correct_answer` — эксплуатационная реальность.

## 0. Базовый принцип: уйти от имён к пространству возможностей

Не «модель = XGBoost», а «модель = вектор возможностей». То же для датасета. Тогда сравнение гетерогенных объектов становится измеримым.

**Вектор возможностей модели (model capability vector):**

```json
{
  "modality": ["tabular","text"],
  "task_types": ["classification","regression"],
  "output_type": "scalar|prob|topk|mask|text|box",
  "supervision": ["supervised","self-supervised"],
  "data_regime": {"min_samples": 100, "few_shot": true, "pretrained": true},
  "context_length": 0,
  "multilingual": false,
  "domain_pretraining": ["general"],
  "latency_class": "low|medium|high",
  "size_class": "small|medium|large",
  "explainability": "native|posthoc|none",
  "deploy_targets": ["server","edge","batch"],
  "license": "permissive|copyleft|commercial|open_weights",
  "hardware_need": "cpu|gpu|tpu",
  "fine_tunable": true,
  "api_available": true
}
```

**Вектор возможностей датасета (dataset card vector):**

```json
{
  "modality": ["text"],
  "task_fit": ["classification","NER"],
  "size": 120000,
  "label_type": "single|multi|sequence|none",
  "label_quality": "gold|silver|weak|unknown",
  "domain": "medical",
  "language": ["ru"],
  "license": "CC-BY",
  "temporal": "2021-2023",
  "pii": false,
  "bias_flags": [],
  "provenance": "web|manual|synthetic",
  "annotation_guideline": true
}
```

Дальше всё — метрики **согласования требований промпта с этими векторами**.

---

## 21. Метрики покрытия каталога (catalog coverage)

Первое, что нужно измерять при гетерогенности, — покрывает ли ваш каталог вообще то, что просят.

| Метрика | Что показывает |
|---|---|
| `catalog_task_coverage` | Доля типов задач из таксономии, на которые есть хотя бы одна модель |
| `catalog_modality_coverage` | Покрытие модальностей |
| `catalog_domain_coverage` | Покрытие доменов |
| `catalog_language_coverage` | Покрытие языков |
| `catalog_regime_coverage` | Покрытие few-shot / zero-shot / full-data |
| `catalog_latency_coverage` | Есть ли модели под каждый класс latency |
| `catalog_explainability_coverage` | Есть ли объяснимые модели в каждой нише |
| `catalog_license_coverage` | Покрытие разрешённых лицензий |
| `catalog_gap_count` | Число ниш без модели |
| `catalog_gap_severity` | Насколько критичны пробелы |
| `prompt_to_catalog_distance` | Ближайшее расстояние от промпта до каталога |
| `empty_bucket_rate` | Доля промптов, попавших в пустую ячейку таксономии |

**Зачем:** если 20% промптов падают в пустые ячейки — вы не «плохо классифицируете», у вас **дыра в каталоге**, и её надо мерить отдельно от точности агента.

---

## 22. Иерархические метрики (промпт → тип задачи → семейство → модель)

Гетерогенность требует иерархии: домен → задача → модальность → семейство → конкретная модель.

| Метрика | Что показывает |
|---|---|
| `hier_accuracy_level1` | Точность на уровне «домен/тип задачи» |
| `hier_accuracy_level2` | Точность на уровне семейства |
| `hier_accuracy_level3` | Точность на уровне конкретной модели |
| `hier_consistency` | Если верно на листе — верно ли на родителе (и наоборот) |
| `parent_child_mismatch_rate` | Доля случаев, где уровни противоречат |
| `hier_precision/recall_per_node` | По каждому узлу отдельно |
| `confusion_at_level1` | Какие типы задач путаются между собой |
| `confusion_at_level2` | Какие семейства путаются |
| `tree_depth_error` | На каком уровне обычно рвётся |
| `abstain_at_level` | На каком уровне агент должен был отказаться |

**Ключевой нюанс:** при гетерогенном каталоге метрика `hier_consistency` часто важнее, чем точность листа. Лучше сказать «это NLP-классификация текста» и дать 5 кандидатов, чем уверенно назвать одну неверную модель.

---

## 23. Open-set метрики (когда нужной модели в каталоге нет)

Обязательный блок, если каталог растёт и разнороден.

| Метрика | Что показывает |
|---|---|
| `novelty_score` | Насколько промпт не похож на обучающие примеры агента |
| `ood_auroc` | Отделяет ли агент «свои» промпты от «чужих» |
| `ood_fpr_at_tpr90` | Ложные срабатывания при 90% обнаружения |
| `unknown_task_detection_f1` | Находит ли задачи вне таксономии |
| `unknown_modality_detection_f1` | Находит ли незнакомые модальности |
| `new_domain_detection_f1` | Находит ли незнакомые домены |
| `rejection_accuracy` | Правильно ли говорит «нет подходящей модели» |
| `false_rejection_rate` | Отказывается, когда модель есть |
| `false_acceptance_rate` | Не отказывается, когда модели нет |
| `softmax_max_prob` | Сырая уверенность top-1 |
| `energy_score` | OOD-скор по логитам |
| `mahalanobis_distance` | Расстояние до ближайшего кластера |
| `knn_distance_to_train` | Близость к обучающим примерам |
| `conformal_set_size` | Размер conformal-набора (адаптивная неуверенность) |
| `conformal_coverage` | Покрытие conformal-интервала |
| `escalation_rate` | Доля эскалаций человеку |
| `escalation_precision` | Эскалирует ли только там, где надо |

**Практика:** без `conformal prediction` и `ood_auroc` агент будет уверенно рекомендовать BERT для задачи, где нужен GNN или симуляция.

---

## 24. Long-tail метрики (редкие модели и редкие задачи)

Гетерогенный каталог почти всегда имеет хвост: 5 популярных семейств и 200 редких.

| Метрика | Что показывает |
|---|---|
| `long_tail_recall@k` | Попадание в редкие модели при большом k |
| `head_vs_tail_accuracy` | Разрыв качества между частыми и редкими задачами |
| `tail_coverage@k` | Доля хвостовых запросов, где ответ в top-k |
| `frequency_bias` | Смещение к популярным моделям |
| `popularity_correlation` | Корреляция рекомендаций с популярностью модели |
| `novelty_of_recommendation` | Насколько рекомендации не банальны |
| `serendipity_proxy` | Полезная неожиданность |
| `tail_abstention_rate` | Как часто агент отказывается на хвосте |
| `per_class_accuracy_min` | Худший класс — важнее среднего |
| `macro_f1_per_family` | Среднее по семействам, а не по объектам |
| `worst_group_accuracy` | Худшая группа (fairness-подход к моделям) |
| `gini_of_recommendations` | Неравномерность распределения рекомендаций |

---

## 25. Метрики гетерогенного ранжирования (сравнение несравнимых моделей)

Когда в top-k смешаны CatBoost, LLM, GNN и симуляция — как сравнивать?

| Метрика | Что показывает |
|---|---|
| `normalized_score` | Каждая метрика приводится к [0,1] через min-max / z-score внутри задачи |
| `rank_score` | Используется ранг, а не абсолютное значение |
| `relative_lift` | Прирост над бейзлайном задачи, а не абсолют |
| `percentile_rank` | Модель в перцентилях по своей задаче |
| `bradley_terry_rating` | Единый рейтинг через попарные сравнения |
| `elo_rating` | Упрощённая версия BT для стриминга |
| `thompson_sampling_regret` | Регрет при исследовании каталога |
| `ucb_score` | Upper Confidence Bound для отбора |
| `score_uncertainty` | Ширина доверительного интервала оценки |
| `interval_coverage` | Покрытие интервала оценки |
| `credible_interval_width` | Байесовская неопределённость |
| `rank_stability` | Стабильность ранга при смене сида/фолда |
| `rank_correlation_spearman` | Согласованность ранжирований |
| `rank_correlation_kendall` | То же, устойчивее к хвостам |
| `ndcg_normalized` | NDCG с нормализацией по задаче |
| `mrr_per_family` | MRR по каждому семейству отдельно |
| `coverage_weighted_ndcg` | NDCG с учётом покрытия |

---

## 26. Метрики portfolio-рекомендации (агент выдаёт набор, а не одну модель)

| Метрика | Что показывает |
|---|---|
| `oracle@k` | Верхняя граница: лучшая модель в top-k |
| `regret@k` | Насколько top-k хуже оракула |
| `coverage@k` | Сколько разных семейств в top-k |
| `diversity@k` | Intra-list diversity (1 − среднее косинусное сходство) |
| `complementarity@k` | Насколько модели дополняют друг друга |
| `redundancy@k` | Дублирование моделей |
| `portfolio_expected_quality` | Ожидаемое качество лучшей в наборе |
| `portfolio_quality_variance` | Разброс качества в наборе |
| `portfolio_risk` | Риск, что все кандидаты плохие |
| `portfolio_calibration` | Калибровка ожидаемого качества набора |
| `hit_at_k` | Есть ли хотя бы одна подходящая модель |
| `all_relevant_coverage` | Покрытие всех подходящих моделей |
| `set_precision` | Доля релевантных в наборе |
| `set_recall` | Доля найденных релевантных |
| `set_f1` | Гармоническое |
| `jaccard_at_k` | Пересечение с эталонным набором |
| `size_efficiency` | Качество на единицу длины набора |

**Смысл:** при гетерогенности правильный ответ — «вот 3–5 разных семейств с обоснованием», а не «вот одна модель».

---

## 27. Метрики dataset-model fit (согласование данных и модели)

Отдельно от выбора модели — соответствует ли датасет модели.

| Метрика | Что показывает |
|---|---|
| `modality_match` | Совпадает ли модальность |
| `label_type_match` | Совпадает ли тип разметки |
| `size_fit` | Достаточно ли данных для модели |
| `size_overkill` | Не избыточна ли модель для таких данных |
| `few_shot_fit` | Подходит ли режим малого числа примеров |
| `language_fit` | Совпадает ли язык |
| `domain_fit` | Совпадает ли домен |
| `license_compatibility` | Совместимы ли лицензии данных и модели |
| `pii_compatibility` | Можно ли использовать данные с PII |
| `quality_fit` | Хватает ли качества разметки для модели |
| `label_noise_tolerance` | Переносит ли модель шум |
| `feature_availability` | Есть ли признаки в проде |
| `temporal_validity` | Актуальны ли данные |
| `schema_match` | Совпадает ли схема |
| `sample_rate_fit` | Совпадает ли частота сэмплирования |
| `class_balance_fit` | Справляется ли модель с дисбалансом |
| `augmentation_need` | Нужна ли аугментация |
| `synthetic_need` | Нужны ли синтетические данные |
| `fit_score` | Интегральный скор соответствия |

---

## 28. Метрики dataset-quality как вход агента

Чтобы оценивать fit, нужны метрики самих данных (считаются автоматически по карточке/профилю).

| Группа | Метрики |
|---|---|
| Полнота | `missing_rate`, `null_ratio`, `feature_coverage`, `label_coverage` |
| Согласованность | `label_conflict_rate`, `duplicate_rate`, `contradiction_rate` |
| Баланс | `class_imbalance_ratio`, `entropy_of_labels`, `long_tail_ratio` |
| Качество разметки | `inter_annotator_agreement` (Cohen κ, Krippendorff α), `guideline_presence` |
| Шум | `label_noise_estimate`, `outlier_rate`, `adversarial_ratio` |
| Утечки | `target_leakage_flag`, `temporal_leakage_flag`, `train_test_overlap` |
| Репрезентативность | `coverage_of_domain`, `sampling_bias_score`, `demographic_balance` |
| Актуальность | `data_age`, `drift_since_collection`, `schema_drift` |
| Безопасность | `pii_density`, `toxicity_rate`, `copyright_risk` |
| Воспроизводимость | `provenance_completeness`, `version_pinned`, `hash_available` |
| Стоимость | `labeling_cost`, `storage_cost`, `refresh_cost` |

---

## 29. Метрики кросс-доменной и кросс-модальной генерализации агента

Раз модели и данные разные — агент должен уметь переносить знания между нишами.

| Метрика | Что показывает |
|---|---|
| `cross_domain_accuracy` | Точность на домене, которого не было в обучении |
| `cross_modality_accuracy` | Точность на новой модальности |
| `cross_task_accuracy` | Точность на новом типе задачи |
| `cross_language_accuracy` | Точность на новом языке |
| `transfer_gap` | Падение качества при переносе |
| `few_shot_adaptation_gain` | Сколько даёт 5–10 примеров |
| `zero_shot_accuracy` | Качество без примеров |
| `cold_start_latency` | Сколько промптов нужно до приемлемого качества |
| `warm_start_gain` | Прирост после N примеров |
| `forgetting_rate` | Забывание старого при обучении новому |
| `backward_compatibility` | Не ломается ли качество на старых промптах |
| `domain_shift_auroc` | Обнаружение сдвига домена |
| `modality_shift_auroc` | Обнаружение сдвига модальности |

---

## 30. Метрики самого каталога (model registry metrics)

Раз каталог большой и разный, он сам объект измерения.

| Метрика | Что показывает |
|---|---|
| `registry_completeness` | Заполнены ли capability-векторы у всех моделей |
| `metadata_quality_score` | Качество метаданных |
| `stale_entry_rate` | Доля устаревших записей |
| `broken_reference_rate` | Битые ссылки, устаревшие API |
| `license_known_rate` | Доля моделей с известной лицензией |
| `benchmark_coverage` | Сколько моделей покрыто бенчмарками |
| `benchmark_recency` | Свежесть бенчмарков |
| `comparable_benchmark_rate` | Доля моделей, сравнимых на общем бенчмарке |
| `onboarding_time_new_model` | Время добавления новой модели |
| `onboarding_error_rate` | Ошибки при онбординге |
| `deduplication_rate` | Доля дублей |
| `taxonomy_stability` | Насколько часто меняется таксономия |
| `taxonomy_drift` | Сдвиг таксономии |
| `catalog_growth_rate` | Скорость роста |

---

## 31. Метрики таксономии (главный риск при гетерогенности)

| Метрика | Что показывает |
|---|---|
| `taxonomy_coverage_of_prompts` | Какой % промптов таксономия объясняет |
| `unmapped_prompt_rate` | Доля промптов вне таксономии |
| `taxonomy_depth_balance` | Не перекошено ли дерево |
| `leaf_distribution_entropy` | Не слишком ли сконцентрировано |
| `overlapping_categories` | Пересекающиеся категории |
| `ambiguous_category_rate` | Неоднозначные категории |
| `category_granularity_score` | Достаточно ли мелкие категории |
| `annotator_agreement_on_taxonomy` | Согласие экспертов по разметке категорий |
| `taxonomy_change_frequency` | Частота изменений |
| `label_consistency_after_change` | Согласованность после изменений |

---

## 32. Метрики решения «строить / брать готовое / не строить»

При гетерогенности агент должен уметь сказать «эту задачу ML не решает».

| Метрика | Что показывает |
|---|---|
| `ml_feasibility_score` | Есть ли вообще ML-постановка |
| `need_for_ml_score` | Нужен ли ML или хватит правил |
| `heuristic_baseline_expected` | Ожидаемое качество простого правила |
| `lift_needed_to_justify_ml` | Порог, ниже которого ML не оправдан |
| `data_sufficiency_score` | Хватает ли данных |
| `label_cost_breakeven` | Когда разметка окупится |
| `build_vs_buy_score` | Строить или брать готовое |
| `buy_vs_api_score` | Готовое ПО или внешнее API |
| `non_ml_solution_hint` | Похоже на задачу без ML |
| `abstain_recommendation` | Агент рекомендует не делать ML |

---

## 33. Метрики эксплуатации агента на гетерогенном каталоге

| Метрика | Что показывает |
|---|---|
| `p50/p95/p99_latency` | Скорость ответа |
| `latency_growth_with_catalog` | Деградация при росте каталога |
| `retrieval_latency` | Скорость поиска по каталогу |
| `index_size_gb` | Размер индекса |
| `index_refresh_time` | Время обновления индекса |
| `cost_per_prompt` | Стоимость анализа |
| `cost_per_correct_answer` | Стоимость правильного ответа |
| `cache_hit_rate` | Попадание в кэш |
| `semantic_cache_hit` | Кэш по эмбеддингам |
| `fallback_rate` | Доля fallback-ответов |
| `fallback_success_rate` | Успешность fallback |
| `degraded_mode_rate` | Работа в деградированном режиме |
| `catalog_update_downtime` | Простой при обновлении каталога |
| `version_compatibility_errors` | Ошибки совместимости версий |

---

## 34. Что логировать по каждому промпту (схема join-таблицы)

Чтобы посчитать всё выше, нужна единая таблица `(prompt_id, model_id)`:

| Поле | Пример |
|---|---|
| `prompt_id`, `prompt_hash` | p-123, sha256 |
| `prompt_features` | JSON с признаками из предыдущих ответов |
| `requirement_vector` | Вектор требований |
| `model_id`, `model_capability_vector` | m-xgb-01, JSON |
| `dataset_id`, `dataset_card` | d-045, JSON |
| `task_type_pred`, `task_type_true` | classification / classification |
| `family_pred`, `family_true` | boosting / boosting |
| `model_rank` | 1 |
| `score`, `confidence`, `interval` | 0.82, 0.71, [0.61, 0.91] |
| `fit_score` | 0.78 |
| `evidence_quote` | «нужно предсказать цену» |
| `assumption_ids` | [a1, a2] |
| `missing_field_ids` | [target_units] |
| `is_ood` | false |
| `abstained` | false |
| `user_feedback` | accepted / rejected / partial |
| `final_choice_model_id` | m-xgb-01 |
| `outcome_quality` | post-hoc: MAE реальной модели |

Это позволяет считать **и офлайн-, и онлайн-метрики** без ручной разметки.

---

## 35. MVP для гетерогенного случая (если надо быстро)

К 12 метрикам первой волны и 12 второй добавьте именно **гетерогенные**:

25. `catalog_task_coverage` и `empty_bucket_rate` — есть ли вообще ответ.  
26. `hier_accuracy_level1/2/3` + `hier_consistency` — не рвётся ли дерево.  
27. `ood_auroc` + `unknown_task_detection_f1` — узнаёт ли «чужое».  
28. `conformal_set_size` + `conformal_coverage` — честная неуверенность.  
29. `long_tail_recall@k` + `worst_group_accuracy` — не забыт ли хвост.  
30. `oracle@k`, `regret@k`, `diversity@k` — качество набора, а не одной модели.  
31. `modality_match`, `size_fit`, `license_compatibility` — dataset-model fit.  
32. `label_noise_estimate`, `inter_annotator_agreement` — качество данных.  
33. `cross_domain_accuracy`, `transfer_gap` — перенос.  
34. `ml_feasibility_score`, `build_vs_buy_score` — не строить ML там, где не надо.  
35. `registry_completeness`, `stale_entry_rate` — здоровье каталога.  
36. `cost_per_correct_answer`, `fallback_rate` — эксплуатация.

---

## 36. Главный вывод

При гетерогенных моделях и датасетах схема меняется так:

1. **Модели и датасеты — не имена, а векторы возможностей.**  
2. **Промпт — не класс, а вектор требований + вектор ограничений.**  
3. **Ответ агента — не одна модель, а ранжированный портфель с интервалами качества.**  
4. **Метрики делятся на три уровня:** покрытие каталога, согласование (fit), качество ранжирования/набора.  
5. **Обязательны open-set и conformal-метрики** — иначе на разнородном каталоге агент будет уверенно ошибаться.  
6. **Обязательны иерархические метрики** — падение на уровне семейства менее критично, чем уверенная неверная конкретная модель.  
7. **Обязательны dataset-quality и dataset-model fit** — половина ошибок «какая модель» на самом деле ошибки «какие данные».

### 🧠 Анализ и классификация промптов
Эти репозитории решают первую часть вашей задачи — превратить текст на естественном языке в структурированный сигнал (тип задачи, сложность, интент).

*   **`VijayV28/Prompt-Classification`**: Исследовательский проект, который сравнивает три подхода к классификации промптов для маршрутизации задач: дообученный BERT, традиционный ML (Logistic Regression на TF-IDF) и промпт-инжиниринг с LLM. Полезен как готовый бенчмарк для выбора метода.
*   **`reddgr/zero-shot-text-classification`**: Скрипты и ноутбуки для дообучения NLI-модели под zero-shot классификацию текста. Подходит, если вы хотите классифицировать промпты по новым, заранее не определённым категориям без сбора размеченных данных.
*   **`nvidia/prompt-task-and-complexity-classifier`**: Многоголовочная модель от NVIDIA, которая классифицирует промпты одновременно по типу задачи (11 категорий) и уровню сложности. Готовый компонент для извлечения ключевых признаков из промпта.
*   **`brave/Androcles`**: Внутренняя модель классификации промптов от Brave, которая распределяет запросы по категориям для динамического изменения функциональности. Репозиторий содержит скрипты для обучения и генерации данных.
*   **`lm-sys/fastchat` (компонент Prompt Classification)**: В рамках проекта FastChat реализован модуль классификации промптов из Chatbot Arena, который использует LLM для категоризации запросов. Пример продакшн-подхода к задаче.

### 🚦 Маршрутизация и выбор LLM (Prompt Routing)
Этот блок — прямой аналог вашей задачи, но для выбора не «классических» ML-моделей, а LLM. Архитектурные решения хорошо переносятся.

*   **`Mattbusel/llm-router`**: Легковесный C++17-роутер, который локально оценивает сложность промпта и выбирает модель по стратегии (цена, скорость, качество). Может обучаться на основе обратной связи (`record_outcome`). Отличный пример простого и быстрого скоринга.
*   **`ES7/routerllm`**: Python-библиотека с аналогичной идеей: маршрутизирует промпты на основе сложности, стоимости или качества. Поддерживает несколько провайдеров (OpenAI, Anthropic, Gemini) и возвращает причину выбора.
*   **`SalesforceAIResearch/xRouter`**: Более сложный research-проект, где маршрутизация LLM обучается с подкреплением (RL) для выбора оптимальной модели среди 20+ кандидатов с учётом производительности и стоимости.
*   **`nordlys` (на PyPI)**: Инструмент, который кластеризует похожие промпты и учится, какая модель лучше работает для каждого кластера, а затем маршрутизирует новые запросы в оптимальную модель. Идея кластеризации промптов напрямую применима к вашей задаче.
*   **`Apofenic/mcp-prompt-router`**: Пример локального роутера, который использует комбинацию regex-эвристик и LLM для классификации и маршрутизации промптов, что может быть дешёвой альтернативой для MVP.

### 🔍 Поиск и рекомендация моделей (RAG-подход)
Поскольку каталог моделей огромен и разнороден, поиск по семантическому сходству — ключевой подход.

*   **`iLoveCmajor/hf-model-finder`**: **Наиболее близкий к вашей задаче репозиторий.** Это RAG-система, которая по описанию ML-задачи на естественном языке ищет подходящую предобученную модель среди ~3,600 карточек Hugging Face. Включает пайплайн инжеста, эмбеддинги, LLM-слой для объяснения выбора и сквозную оценку качества.
*   **`neulab/prompt2model`**: Академический проект (от NeuLab, CMU), который по промпту автоматически генерирует язык��вую модель. Включает компоненты **Prompt Parser**, **Dataset Retriever**, **Dataset Generator** и **Model Retriever**. Репозиторий ценен как референсная архитектура полного пайплайна «промпт → данные → модель».

### ⚙️ AutoML и выбор моделей (классический ML)
Если часть вашего каталога — это табличные данные и классические модели, эти фреймворки решают задачу выбора алгоритма и гиперпараметров.

*   **`automl/auto-sklearn`**: Зрелый AutoML-фреймворк, который автоматически выбирает модели и гиперпараметры для задач на табличных данных, будучи drop-in заменой оценщика scikit-learn.
*   **`automl/autoweka`**: Аналог auto-sklearn, но для экосистемы WEKA (Java). Выполняет совместный выбор алгоритма и оптимизацию гиперпараметров.
*   **`EpistasisLab/tpot`**: AutoML-инструмент, который оптимизирует не просто модель, а целые пайплайны предобработки и модели с помощью генетического программирования. Полезен, если важен не только выбор модели, но и feature engineering.
*   **`HDI-Project/ATM`**: (Auto Tune Models) Мульти-тенантная система для автоматического выбора и настройки моделей. Хороший пример архитектуры для многопользовательского сценария.
*   **`microsoft/FLAML`**, **`pycaret/pycaret`**, **`autogluon/autogluon`**: Хотя они не попали в результаты поиска напрямую, это мощные и популярные альтернативы, которые стоит изучить для блока AutoML.

### 📊 Поиск и рекомендация датасетов
Вторая половина вашей задачи — найти не только модель, но и данные для неё.

*   **`viswavi/datafinder`**: Репозиторий системы **DataFinder** для рекомендации научных датасетов. Использует двухэнкодерный ретривер для поиска релевантных датасетов по описанию задачи. Прямой аналог задачи подбора данных.
*   **`ISG-Siegen/FINALLY`**: Веб-система для выбора датасетов в экспериментах по рекомендательным системам. Комбинирует метаданные датасетов с «пространством эффективности алгоритмов» (APS), чтобы рекомендовать наборы данных, соответствующие ограничениям пользователя.
*   **`RUCAIBox/RecSysDatasets`**: Репозиторий с коллекцией публичных источников данных для рекомендательных систем. Полезен как курируемый каталог, по которому можно строить поиск.
*   **`Maqif14/Textual_GCL`**: Реализация системы рекомендации датасетов на основе графов и текстовых признаков с контрастивным обучением. Более исследовательский, но релевантный подход.

### 🧬 Мета-обучение для выбора моделей
Подход, при котором система учится выбирать модель на основе мета-признаков задачи, а не только по текстовому описанию.

*   **`MetaOD`**: Первый мета-обучаемый подход к выбору модели для задачи обнаружения аномалий (Outlier Detection). Использует мета-обучение для выбора лучшей модели на основе характеристик данных.
*   **`Hydra`**: Мета-рекомендатель для выбора модели в задаче обнаружения аномалий в многомерных временных рядах. Демонстрирует, как можно строить рекомендатели для специфических доменов.

### 🧰 Смежные и вспомогательные инструменты
*   **`RUCAIBox/RecBole` (и RecBole-MetaRec)**: Унифицированная библиотека для рекомендательных систем, включающая компоненты для мета-обучения. Может быть полезна, если ваши рекомендации моделей/данных нужно строить как рекомендательную систему.
*   **`bmdietrich/agent-cloud-architect`**: Пример AI-агента, который анализирует проект и рекомендует архитектуру для AWS/GCP/Azure. Хотя это про облачную архитектуру, а не ML-модели, паттерн «агент анализирует входные данные и рекомендует из каталога» полностью совпадает.

### 💎 Как это собрать вместе
Для вашей задачи «один промпт → ML-модель + датасет» наиболее релевантная комбинация выглядит так:

1.  **Анализ промпта**: Взять `VijayV28/Prompt-Classification` или `nvidia/prompt-task-and-complexity-classifier` как основу для извлечения признаков (тип задачи, сложность, модальность).
2.  **Поиск моделей**: Использовать архитектуру `iLoveCmajor/hf-model-finder` для RAG-поиска по каталогу моделей на основе извлечённых признаков.
3.  **Поиск данных**: Интегрировать подход `viswavi/datafinder` для поиска подходящих датасетов.
4.  **Маршрутизация/Ранжирование**: Применить идеи из `routerllm` или `nordlys` для финального выбора и ранжирования кандидатов с учётом ограничений (latency, стоимость, explainability).
